# Lab 2: 진단 에이전트

## 개요
CRM 애플리케이션 스택의 CloudWatch 로그와 지표를 분석하는 Strands 기반 진단 에이전트를 구축합니다.

## 목표
- 인시던트 분석을 위한 Strands 에이전트 생성
- CloudWatch 로그와 지표를 가져오는 사용자 지정 도구 구현
- EC2, DynamoDB 및 NGINX 로그를 가져오는 도구 구축
- 실제 애플리케이션 로그를 대상으로 에이전트 테스트
- 진단 정확도 검증

## 학습 내용
- 도구를 사용하는 Strands 에이전트 구축 방법
- AWS Lambda 기반 도구 생성 방법
- 프로그래밍 방식으로 CloudWatch 로그를 분석하는 방법
- 에이전트 진단 워크플로와 추론

## 아키텍처 개요

```
┌─────────────────┐         ┌──────────────────────┐         ┌─────────────────────┐
│  User Request   │────────▶│   Strands Agent      │────────▶│  AgentCore Gateway  │
│  (Diagnostic    │         │   (Diagnostics)      │         │  (MCP Protocol)     │
│   Query)        │         └──────────────────────┘         └─────────────────────┘
└─────────────────┘                    │                               │
                                       │                               │
                                       ▼                               ▼
                          ┌────────────────────────┐      ┌──────────────────────┐
                          │  Diagnostic Tools      │      │  Lambda Function     │
                          │  ├─ EC2 Logs           │      │  (ZIP Deployment)    │
                          │  ├─ NGINX Logs         │      │  ├─ Session Mgmt     │
                          │  ├─ DynamoDB Metrics   │      │  ├─ Tool Execution   │
                          │  └─ CloudWatch CPU/    │      │  └─ Error Handling   │
                          │     Memory Metrics     │      └──────────────────────┘
                          └────────────────────────┘                │
                                       │                             │
                                       ▼                             ▼
                          ┌────────────────────────┐      ┌──────────────────────┐
                          │  AWS Services          │◀─────│  Analysis Results    │
                          │  ├─ CloudWatch Logs    │      │  & Validation        │
                          │  ├─ EC2 Instances      │      └──────────────────────┘
                          │  ├─ DynamoDB Tables    │
                          │  └─ CloudWatch Metrics │
                          └────────────────────────┘
```

### 주요 구성 요소:

• **다중 도구 오케스트레이션**: 에이전트가 7개의 진단 도구(EC2, NGINX, DynamoDB, CloudWatch)를 조율합니다.
• **안전한 실행**: VPC 액세스 및 IAM 기반 인증을 사용하는 Lambda 함수
• **실시간 분석**: 실시간 로그 및 지표 검색을 위한 CloudWatch 통합

## 0. 필수 패키지 설치

모든 종속성이 설치되도록 이 셀을 먼저 실행합니다.

In [ ]:
%pip install -q -r requirements.txt
print("✅ Workshop dependencies installed")

## 1. 필수 모듈 가져오기

In [ ]:
# AWS SDK 및 구성
import boto3
import json
import datetime

# 워크숍 구성
from lab_helpers.config import MODEL_ID, AWS_REGION, AWS_PROFILE
from lab_helpers.constants import PARAMETER_PATHS

# AWS 클라이언트 초기화
cloudwatch_client = boto3.client("logs", region_name=AWS_REGION)
ec2_client = boto3.client("ec2", region_name=AWS_REGION)
lambda_client = boto3.client("lambda", region_name=AWS_REGION)
sts_client = boto3.client("sts", region_name=AWS_REGION)
agent_memory_client = boto3.client("bedrock-agentcore", region_name=AWS_REGION)

from lab_helpers.lab_01.fault_injection import initialize_fault_injection
from lab_helpers.parameter_store import put_parameter, get_parameter

# AWS 클라이언트를 초기화하고 SSM에서 인프라 리소스 ID 가져오기
print("Initializing fault injection utilities...")
resources = initialize_fault_injection(AWS_REGION, AWS_PROFILE)

print("\nDiscovered Infrastructure Resources:")

print(f"  Nginx Instance: {resources.get('nginx_instance_id', 'Not found')}")
print(f"  App Instance: {resources.get('app_instance_id', 'Not found')}")
print(f"  CRM Activities Table: {resources.get('crm_activities_table_name', 'Not found')}")
print(f"  CRM Customers Table: {resources.get('crm_customers_table_name', 'Not found')}")
print(f"  CRM Deals Table: {resources.get('crm_deals_table_name', 'Not found')}")

print("✅ Imports loaded")

## 2. 사전 요구 사항 확인

In [ ]:
# 사전 요구 사항이 준비되었는지 확인
try:
    identity = sts_client.get_caller_identity()
    account_id = identity["Account"]

    from strands import Agent

    print(f"✅ Prerequisites verified: AWS Account {account_id}, bedrock-agentcore + strands available")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Please ensure AWS credentials are configured and all packages are installed.")

## 3. 헬퍼 도구 생성

In [ ]:
from lab_helpers import mock_data

# CloudWatch에서 로그와 지표를 가져오는 헬퍼 도구 함수


def fetch_crm_app_logs(log_group_name="/aws/sre-workshop/crm-application", hours=2, use_mock=False):
    """CloudWatch에서 CRM 애플리케이션 로그를 가져옵니다."""
    if use_mock:
        return mock_data.get_ec2_logs()

    try:
        now = datetime.datetime.now(datetime.timezone.utc)
        start_time = int((now - datetime.timedelta(hours=hours)).timestamp() * 1000)
        end_time = int(now.timestamp() * 1000)

        response = cloudwatch_client.filter_log_events(
            logGroupName=log_group_name,
            startTime=start_time,
            endTime=end_time,
            filterPattern="?error ?throttle",
            limit=500,
        )
        return response.get("events", [])
    except Exception as e:
        return [{"message": f"Error fetching EC2 logs: {str(e)}"}]


def fetch_ec2_logs(log_group_name="/aws/sre-workshop/application", hours=2, use_mock=False):
    """CloudWatch에서 EC2 애플리케이션 로그를 가져옵니다."""
    if use_mock:
        return mock_data.get_ec2_logs()

    try:
        now = datetime.datetime.now(datetime.timezone.utc)
        start_time = int((now - datetime.timedelta(hours=hours)).timestamp() * 1000)
        end_time = int(now.timestamp() * 1000)

        response = cloudwatch_client.filter_log_events(
            logGroupName=log_group_name,
            startTime=start_time,
            endTime=end_time,
            filterPattern="?error ?throttle",
            limit=500,
        )
        return response.get("events", [])
    except Exception as e:
        return [{"message": f"Error fetching EC2 logs: {str(e)}"}]


def fetch_nginx_error_logs(log_group_name="/aws/sre-workshop/nginx/error", hours=2, use_mock=False):
    """CloudWatch에서 NGINX 오류 로그를 가져옵니다."""
    if use_mock:
        return mock_data.get_nginx_logs()

    try:
        now = datetime.datetime.now(datetime.timezone.utc)
        start_time = int((now - datetime.timedelta(hours=hours)).timestamp() * 1000)
        end_time = int(now.timestamp() * 1000)

        response = cloudwatch_client.filter_log_events(
            logGroupName=log_group_name,
            startTime=start_time,
            endTime=end_time,
            filterPattern="?error ?throttle",
            limit=500,
        )
        return response.get("events", [])
    except Exception as e:
        return [{"message": f"Error fetching NGINX error logs: {str(e)}"}]


def fetch_nginx_access_logs(log_group_name="/aws/sre-workshop/nginx/access", hours=24, use_mock=False):
    """CloudWatch에서 NGINX 액세스/오류 로그를 가져옵니다."""
    if use_mock:
        return mock_data.get_nginx_logs()

    try:
        now = datetime.datetime.now(datetime.timezone.utc)
        start_time = int((now - datetime.timedelta(hours=hours)).timestamp() * 1000)
        end_time = int(now.timestamp() * 1000)

        response = cloudwatch_client.filter_log_events(
            logGroupName=log_group_name,
            startTime=start_time,
            endTime=end_time,
            limit=100,
        )
        return response.get("events", [])
    except Exception as e:
        return [{"message": f"Error fetching NGINX access logs: {str(e)}"}]


def fetch_dynamodb_metrics(table_name, period_minutes=60, use_mock=False):
    """CloudWatch에서 DynamoDB 작업 로그를 가져옵니다."""
    if use_mock:
        return mock_data.get_dynamodb_logs()

    try:
        end_time = datetime.datetime.utcnow()
        start_time = end_time - datetime.timedelta(minutes=period_minutes)

        # get_metric_data를 사용하여 한 번의 호출로 모든 지표 쿼리
        cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
        response = cloudwatch.get_metric_data(
            MetricDataQueries=[
                {
                    "Id": "read_capacity",
                    "MetricStat": {
                        "Metric": {
                            "Namespace": "AWS/DynamoDB",
                            "MetricName": "ConsumedReadCapacityUnits",
                            "Dimensions": [{"Name": "TableName", "Value": table_name}],
                        },
                        "Period": 300,
                        "Stat": "Sum",
                    },
                },
                {
                    "Id": "write_capacity",
                    "MetricStat": {
                        "Metric": {
                            "Namespace": "AWS/DynamoDB",
                            "MetricName": "ConsumedWriteCapacityUnits",
                            "Dimensions": [{"Name": "TableName", "Value": table_name}],
                        },
                        "Period": 300,
                        "Stat": "Sum",
                    },
                },
                {
                    "Id": "throttled",
                    "MetricStat": {
                        "Metric": {
                            "Namespace": "AWS/DynamoDB",
                            "MetricName": "ThrottledRequests",
                            "Dimensions": [{"Name": "TableName", "Value": table_name}],
                        },
                        "Period": 300,
                        "Stat": "Sum",
                    },
                },
                {
                    "Id": "user_errors",
                    "MetricStat": {
                        "Metric": {
                            "Namespace": "AWS/DynamoDB",
                            "MetricName": "UserErrors",
                            "Dimensions": [{"Name": "TableName", "Value": table_name}],
                        },
                        "Period": 300,
                        "Stat": "Sum",
                    },
                },
                {
                    "Id": "system_errors",
                    "MetricStat": {
                        "Metric": {
                            "Namespace": "AWS/DynamoDB",
                            "MetricName": "SystemErrors",
                            "Dimensions": [{"Name": "TableName", "Value": table_name}],
                        },
                        "Period": 300,
                        "Stat": "Sum",
                    },
                },
                {
                    "Id": "latency",
                    "MetricStat": {
                        "Metric": {
                            "Namespace": "AWS/DynamoDB",
                            "MetricName": "SuccessfulRequestLatency",
                            "Dimensions": [{"Name": "TableName", "Value": table_name}],
                        },
                        "Period": 300,
                        "Stat": "Average",
                    },
                },
            ],
            StartTime=start_time,
            EndTime=end_time,
        )

        # 응답에서 값 추출
        result = {
            "table_name": table_name,
            "timestamp": end_time.isoformat(),
            "read_capacity": 0,
            "write_capacity": 0,
            "throttled_requests": 0,
            "user_errors": 0,
            "system_errors": 0,
            "avg_latency_ms": None,
        }

        for metric_result in response["MetricDataResults"]:
            metric_id = metric_result["Id"]
            values = metric_result["Values"]

            if values:
                if metric_id == "latency":
                    result["avg_latency_ms"] = sum(values) / len(values)
            else:
                result[metric_id.replace("_", "_")] = sum(values)

        return result
    except Exception as e:
        return [{"message": f"Error fetching DynamoDB logs: {str(e)}"}]


def get_cpu_metrics(instance_id, period_minutes=60):
    """CloudWatch metric을 가져오는 헬퍼 함수입니다."""
    cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
    end_time = datetime.datetime.now(datetime.UTC)
    start_time = end_time - datetime.timedelta(minutes=period_minutes)

    response = cloudwatch.get_metric_data(
        MetricDataQueries=[
            {
                "Id": "m1",
                "MetricStat": {
                    "Metric": {
                        "Namespace": "AWS/EC2",
                        "MetricName": "CPUUtilization",
                        "Dimensions": [{"Name": "InstanceId", "Value": instance_id}],
                    },
                    "Period": 60,
                    "Stat": "Average",
                },
            }
        ],
        StartTime=start_time,
        EndTime=end_time,
    )

    values = response["MetricDataResults"][0]["Values"]
    return values[-1] if values else None


def get_memory_metrics(instance_id, period_minutes=60):
    """CloudWatch metric을 가져오는 헬퍼 함수입니다."""
    cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
    end_time = datetime.datetime.now(datetime.UTC)

    start_time = end_time - datetime.timedelta(minutes=period_minutes)

    response = cloudwatch.get_metric_data(
        MetricDataQueries=[
            {
                "Id": "m1",
                "MetricStat": {
                    "Metric": {
                        "Namespace": "AWS/EC2",
                        "MetricName": "mem_used_percent",
                        "Dimensions": [{"Name": "InstanceId", "Value": instance_id}],
                    },
                    "Period": 60,
                    "Stat": "Average",
                },
            }
        ],
        StartTime=start_time,
        EndTime=end_time,
    )

    values = response["MetricDataResults"][0]["Values"]
    return values[-1] if values else None


print("✅ Helper tools defined")

## 4. 헬퍼 도구 테스트

In [ ]:
# 섹션 3에서 생성한 헬퍼 도구 테스트

# 1. 애플리케이션 리소스 가져오기
nginx_instance_id = resources.get("nginx_instance_id")
app_instance_id = resources.get("app_instance_id")
crm_activities_table_name = resources.get("crm_activities_table_name")
crm_customers_table_name = resources.get("crm_customers_table_name")
crm_deals_table_name = resources.get("crm_deals_table_name")

# 2. 몇 가지 값으로 도구 테스트
crm_app_logs = fetch_crm_app_logs()
print(crm_app_logs)
ec2_logs = fetch_ec2_logs()
print(ec2_logs)
nginx_error_logs = fetch_nginx_error_logs()
print(nginx_error_logs)
nginx_access_logs = fetch_nginx_access_logs()
print(nginx_access_logs)
ddb_logs = fetch_dynamodb_metrics(table_name=crm_customers_table_name)

print(ddb_logs)

cpu_metrics = get_cpu_metrics(instance_id=nginx_instance_id)
print(cpu_metrics)
memory_metrics = get_memory_metrics(instance_id=nginx_instance_id)
print(memory_metrics)

# print(f"✅ Helper tools verified: EC2({len(ec2_logs)}), NGINX({len(nginx_logs)}), DynamoDB({len(ddb_logs)}), CPU({len(cpu_metrics)}), Memory({len(memory_metrics)})")

## 5. 로컬에 Strands 프레임워크 추가

**목표:** 지능형 에이전트 기반 추론을 위해 진단 도구를 Strands와 통합합니다.

**접근 방식:** Tool 객체를 정의하고 에이전트 인스턴스를 생성한 후 로컬에서 엔드 투 엔드로 테스트합니다.

**핵심 학습 내용:** Strands가 복잡한 진단을 위해 도구 사용을 오케스트레이션하는 방법

In [ ]:
### 5.1: Strands 도구 정의

from strands import tool

table_names = [key for key in resources.keys() if key.endswith("_table_name") and "crm" in key]
nginx_instance_id = resources.get("nginx_instance_id")
app_instance_id = resources.get("app_instance_id")
crm_activities_table_name = resources.get("crm_activities_table_name")
crm_customers_table_name = resources.get("crm_customers_table_name")
crm_deals_table_name = resources.get("crm_deals_table_name")


# 도구 1: CRM 애플리케이션 로그
@tool(description="Fetch CRM application logs to identify application errors and issues")
def get_crm_app_logs(limit: int = 10):
    """Fetch recent crm application logs"""
    crm_app_logs = fetch_crm_app_logs()
    return crm_app_logs


# 도구 2: EC2 로그
@tool(description="Fetch EC2 application logs to identify application errors and issues")
def get_ec2_logs(limit: int = 10):
    """Fetch recent EC2 application logs"""
    ec2_logs = fetch_ec2_logs()
    return ec2_logs


# 도구 3: NGINX 오류 로그
@tool(description="Fetch NGINX error logs")
def get_nginx_error_logs():
    """Fetch NGINX error logs"""
    nginx_error_logs = fetch_nginx_error_logs()
    return nginx_error_logs


# 도구 4: NGINX 액세스 로그
@tool(description="Fetch NGINX access logs")
def get_nginx_access_logs():
    """Fetch NGINX access/error logs"""
    nginx_access_logs = fetch_nginx_access_logs()
    return nginx_access_logs


# 도구 5: DynamoDB 지표
@tool(description="Fetch DynamoDB metrics to detect throttling and service issues")
def get_dynamodb_metrics():
    """Fetch DynamoDB operation metrics"""
    ddb_metrics = ""
    for table in table_names:
        ddb_metrics += str(fetch_dynamodb_metrics(table_name=table))

    return ddb_metrics


# 도구 6: 애플리케이션 CPU 지표
@tool(description="Fetch CloudWatch metrics (CPU) to analyze resource utilization for an instance")
def get_cloudwatch_cpu_metrics():
    """Fetch CloudWatch CPU metrics"""
    cpu_metrics = get_cpu_metrics(instance_id=nginx_instance_id) + get_cpu_metrics(instance_id=app_instance_id)
    return cpu_metrics


# 도구 7: 애플리케이션 메모리 지표
@tool(description="Fetch CloudWatch metrics (memory) to analyze resource utilization an instance")
def get_cloudwatch_memory_metrics():
    """Fetch CloudWatch memory metrics"""
    memory_metrics = get_memory_metrics(instance_id=nginx_instance_id) + get_memory_metrics(instance_id=app_instance_id)
    return memory_metrics


print("✅ Strands tools defined (7 tools)")
print("   • get crm logs")
print("   • get ec2 logs")
print("   • get nginx error logs")
print("   • get nginx accesslogs")
print("   • get dynamodb metrics")
print("   • get cloudwatch cpu metrics")
print("   • get cloudwatch memory metrics")

In [ ]:
### 5.2: Strands 에이전트 생성

from strands import Agent

# 진단 도구를 사용하는 에이전트 생성
diagnostic_agent = Agent(
    name="system_diagnostics_agent",
    description="Expert system diagnostics agent for analyzing logs and metrics",
    model=MODEL_ID,
    tools=[
        get_crm_app_logs,
        get_ec2_logs,
        get_nginx_error_logs,
        get_nginx_access_logs,
        get_dynamodb_metrics,
        get_cloudwatch_cpu_metrics,
        get_cloudwatch_memory_metrics,
    ],
    system_prompt="""
    You are an expert system diagnostics agent. Your role is to analyze system logs and metrics to identify issues and their high level root causes(including AWS resources such as ARNs,IDs etc. causing them).

When diagnosing system issues:
1. Start by gathering relevant logs (EC2, NGINX, DynamoDB)
2. Check CloudWatch metrics to understand resource utilization patterns
3. Correlate findings across and provide a fairly detailed but consize assessment with severity
4. Once the analysis is complete, in the end share the data sources or points(EC2s, tables etc.), based on which these insights were generated. 
""",
)

print("✅ Strands agent created")
print("   Agent: system_diagnostics_agent")
print(f"   Model: {MODEL_ID}")
print("   Tools: 7 (EC2, NGINX, DynamoDB, CloudWatch)")

## 6. 로컬에서 Strands 에이전트 테스트

**목표:** 모의 데이터로 에이전트 추론과 도구 오케스트레이션을 검증합니다.

**접근 방식:** 진단 쿼리로 에이전트를 호출하고 도구 호출과 추론을 추적합니다.

**핵심 학습 내용:** 에이전트가 문제 해결을 위해 도구를 선택하고 조합하는 방법

In [ ]:
### 6.1: 진단 쿼리로 에이전트 테스트


print("🧪 Testing Strands Agent Locally\n")
print("=" * 70)

test_queries = ["What critical issues do you see in the system? Provide a summary with key data points."]
diagnostics_agent_response = ""


async def test_agent():
    responses = []
    for i, query in enumerate(test_queries, 1):
        print(f"\n[Query {i}] {query}\n")

        try:
            # 에이전트 실행(비동기 호출)
            response = await diagnostic_agent.invoke_async(query)
            print(f"Agent Response:\n{response}\n")
            responses.append(response.message["content"][0]["text"])
        except Exception as e:
            print(f"❌ Error: {e}\n")
            responses.append(None)

    return responses


# Jupyter에서 비동기 테스트 실행
diagnostics_agent_response = await test_agent()

print("=" * 70)
print("✅ Agent test complete")

## 진단 분석을 AgentCore Memory에 기록

In [ ]:
nginx_instance_id = resources.get("nginx_instance_id")
app_instance_id = resources.get("app_instance_id")
crm_activities_table_name = resources.get("crm_activities_table_name")
crm_customers_table_name = resources.get("crm_customers_table_name")
crm_deals_table_name = resources.get("crm_deals_table_name")


memory_id = get_parameter(PARAMETER_PATHS["memory"]["memory_id"])
memory_session_id = get_parameter(PARAMETER_PATHS["memory"]["default_session_id"])

print(memory_id)
print(memory_session_id)
actor_id = "diagnostics_agent"

# 메시지로 페이로드 구성

payload = []
payload.append(
    {
        "conversational": {
            "content": {"text": "nginx EC2 instance id: " + nginx_instance_id + "."},
            "role": "ASSISTANT",
        }
    }
)
payload.append(
    {
        "conversational": {
            "content": {"text": "application EC2 instance id: " + app_instance_id + "."},
            "role": "ASSISTANT",
        }
    }
)
payload.append(
    {
        "conversational": {
            "content": {"text": "CRM Activities Table Name (DynamoDB): " + crm_activities_table_name + "."},
            "role": "ASSISTANT",
        }
    }
)
payload.append(
    {
        "conversational": {
            "content": {"text": "CRM Customers Table Name (DynamoDB): " + crm_customers_table_name + "."},
            "role": "ASSISTANT",
        }
    }
)
payload.append(
    {
        "conversational": {
            "content": {"text": "CRM Deals Table Name (DynamoDB): " + crm_deals_table_name + "."},
            "role": "ASSISTANT",
        }
    }
)
payload.append(
    {
        "conversational": {
            "content": {"text": "diagnostics agent analysis: " + str(diagnostics_agent_response)},
            "role": "ASSISTANT",
        }
    }
)

# 제공된 타임스탬프 또는 현재 UTC 시간 사용
event_timestamp = datetime.datetime.now()

# 요청 파라미터 구성
params = {
    "memoryId": memory_id,
    "actorId": actor_id,
    "sessionId": memory_session_id,
    "eventTimestamp": event_timestamp,
    "payload": payload,
}

response = agent_memory_client.create_event(**params)

# 성공적으로 기록되었는지 확인하기 위해 에이전트 메모리에 추가된 이벤트 나열
params = {
    "memoryId": memory_id,
    "actorId": actor_id,
    "sessionId": memory_session_id,
    "includePayloads": True,
}

response = agent_memory_client.list_events(**params)

for event in response.get("events", []):
    event_id = event.get("eventId")
    # print(f"\nEvent: {event_id}")

    # 모든 메시지 가져오기
    payload = event.get("payload", [])
    for i, item in enumerate(payload):
        if "conversational" in item:
            text = item["conversational"]["content"]["text"]
            role = item["conversational"]["role"]
            print(f"  Message {i}: [{actor_id}] {text}")

## 7. Strands Lambda 핸들러 생성

**목표:** AgentCore Gateway 호출을 위해 Strands 에이전트를 래핑하는 Lambda 핸들러를 생성합니다.

**접근 방식:** Gateway 이벤트 컨텍스트를 수신하고 Strands 에이전트를 호출한 뒤 구조화된 응답을 반환하는 핸들러를 구축합니다.

**핵심 학습 내용:** Strands 에이전트를 Lambda/Gateway 인프라와 연결하는 방법

## 8. Lambda에 Strands 에이전트 배포(ZIP 기반 - VPC 호환)

**목표:** 검증된 Strands 에이전트 코드를 ZIP 패키징을 사용하여 Lambda에 배포합니다.

**사전 요구 사항:** 섹션 5~7을 완료하고 로컬에서 테스트해야 합니다.

In [ ]:
print(AWS_REGION)

In [ ]:
### 8.1: Lambda 배포
!chmod +x lab_helpers/lab_02/deploy.sh
!lab_helpers/lab_02/deploy.sh

## 9. AgentCore Gateway 생성 및 Lambda 대상 등록

**목표:** 배포된 Lambda 함수와 연동할 Gateway 인프라를 설정합니다.

**접근 방식:** 명시적으로 제어할 수 있도록 Boto3를 사용해 단계별로 구축합니다.

**사전 요구 사항:** 섹션 8의 Lambda 배포를 완료해야 합니다.

**핵심 학습 내용:** Gateway가 IAM 인증을 통해 Lambda 함수의 도구 호출을 오케스트레이션하는 방법

**아키텍처:**
```
User Request
    ↓
MCP Client (Section 10)
    ↓ (IAM auth)
Gateway (Section 9)
    ↓ (Gateway service role)
Lambda (Section 8)
    ↓
Strands Agent (Sections 5-7)
```

In [ ]:
### 9.0: Gateway 서비스 역할 생성

print("📋 Setting up Gateway service role...\n")

from lab_helpers.lab_02.gateway_setup import create_gateway_service_role
from lab_helpers.config import AWS_REGION

# Gateway용 IAM 서비스 역할 생성
gateway_role_config = create_gateway_service_role(region_name=AWS_REGION)

print("\n✅ Gateway service role ready")
print(f"   Role ARN: {gateway_role_config['role_arn']}")
print("   Permissions: Lambda invocation + CloudWatch logs")

# 9.1에서 사용할 수 있도록 저장
gateway_role_arn = gateway_role_config["role_arn"]

In [ ]:
### 9.1: Gateway 생성

import boto3
from lab_helpers.config import AWS_REGION

# AgentCore 클라이언트 초기화
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

print("📋 Creating AgentCore Gateway...")

try:
    # Parameter Store에서 Cognito 구성 가져오기
    from lab_helpers.parameter_store import get_parameter
    from lab_helpers.constants import PARAMETER_PATHS

    user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"], region_name=AWS_REGION)
    user_auth_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"], region_name=AWS_REGION)

    # 검색 URL 구성(Lab 3 및 4와 동일한 패턴)
    discovery_url = f"https://cognito-idp.{AWS_REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration"

    # CUSTOM_JWT 권한 부여자를 사용하는 Gateway 생성(Lab 3 및 4와 동일)
    gateway = agentcore_client.create_gateway(
        name="aiml301-diagnostics-gateway",
        roleArn=gateway_role_arn,  # Gateway 서비스 역할(9.0에서 생성)
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",  # 호출자는 Cognito JWT를 사용하여 Gateway 호출
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": discovery_url,
                "allowedClients": [user_auth_client_id],
            }
        },
    )

    gateway_id = gateway["gatewayId"]
    gateway_url = gateway["gatewayUrl"]
    gateway_role_arn_actual = gateway["roleArn"]

    print("✅ Gateway created successfully")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    print(f"   Authorization: CUSTOM_JWT (Cognito User Pool: {user_pool_id})")
    print(f"   Allowed Clients: {user_auth_client_id}")
    print(f"   Service Role: {gateway_role_arn_actual}")
    print("      (used by Gateway to invoke Lambda targets)")

    # Lab 5에서 사용할 Gateway 구성을 Parameter Store에 저장
    from lab_helpers.parameter_store import put_parameter

    put_parameter("/aiml301/lab-02/gateway-id", gateway_id, region_name=AWS_REGION)
    put_parameter("/aiml301/lab-02/gateway-url", gateway_url, region_name=AWS_REGION)
    print("✅ Gateway configuration saved to Parameter Store")

    # 이 Notebook에서 나중에 사용할 수 있도록 저장
    gateway_config = {
        "gateway_id": gateway_id,
        "gateway_url": gateway_url,
        "region": AWS_REGION,
    }

except Exception as e:
    print(f"❌ Error: {e}")
    raise

In [ ]:
print("📝 Defining tool schema for Strands diagnostics agent...\n")

# Gateway는 단일 상위 수준 도구인 "invoke_diagnostics_agent"를 노출
# 이 도구는 자연어 쿼리를 받아 Lambda에서 실행 중인 Strands 에이전트를 호출
# 에이전트는 내부적으로 4개의 진단 도구(EC2, NGINX, DynamoDB, CloudWatch)를 오케스트레이션

tool_schema = [
    {
        "name": "invoke_diagnostics_agent",
        "description": "Invoke the diagnostics agent to analyze system logs and metrics. The agent will orchestrate multiple diagnostic tools (EC2 logs, NGINX logs, DynamoDB logs, CloudWatch metrics) to identify issues and root causes.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural language diagnostic query (e.g., 'What are the main issues?', 'Analyze CPU and memory utilization')",
                }
            },
            "required": ["query"],
        },
    }
]

print("✅ Tool schema defined (1 high-level tool)")
print("   Tool: invoke_diagnostics_agent")
print("   Purpose: Natural language diagnostic queries")
print("   Internal: Agent orchestrates 4 diagnostic tools (EC2, NGINX, DynamoDB, CloudWatch)")
print("\n📐 Architecture:")
print("   Gateway Tool: invoke_diagnostics_agent (natural language interface)")
print("        ↓")
print("   Lambda Handler: Receives query, invokes Strands agent")
print("        ↓")
print("   Strands Agent: Orchestrates local tools")
print("        ├─ get_ec2_logs")
print("        ├─ get_nginx_logs")
print("        ├─ get_dynamodb_logs")
print("        └─ get_cloudwatch_metrics")

In [ ]:
import time

time.sleep(10)

In [ ]:
### 9.3: Lambda 함수를 Gateway 대상으로 등록

print("🔗 Registering Lambda as tool target...\n")

# Parameter Store에서 Lambda 함수 ARN 가져오기
ssm_client = boto3.client("ssm", region_name=AWS_REGION)
lambda_function_arn = ssm_client.get_parameter(Name="/aiml301/lab-02/lambda-function-arn")["Parameter"]["Value"]

print(f"Lambda ARN: {lambda_function_arn}\n")

try:
    # Lambda를 도구 대상으로 등록
    target = agentcore_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="strands-diagnostics-agent",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": lambda_function_arn,
                    "toolSchema": {
                        "inlinePayload": tool_schema  # 이 대상에서 사용할 수 있는 도구 정의
                    },
                }
            }
        },
        credentialProviderConfigurations=[
            {
                "credentialProviderType": "GATEWAY_IAM_ROLE"  # Gateway의 IAM 역할로 Lambda 호출
            }
        ],
    )

    target_id = target["targetId"]

    print("✅ Lambda target registered successfully")
    print(f"   Target ID: {target_id}")
    print("   Target Name: strands-diagnostics-agent")
    print(f"   Lambda ARN: {lambda_function_arn}")
    print("   Tools: 4 (get_ec2_logs, get_nginx_logs, get_dynamodb_logs, get_cloudwatch_metrics)")
    print("   Credentials: GATEWAY_IAM_ROLE (Lambda invoked with Gateway service role)")

    # 대상 정보로 Gateway 구성 업데이트
    gateway_config["target_id"] = target_id
    gateway_config["lambda_arn"] = lambda_function_arn

    print("\n📊 Gateway Configuration Summary:")
    print(f"   Gateway ID: {gateway_config['gateway_id']}")
    print(f"   Gateway URL: {gateway_config['gateway_url']}")
    print(f"   Target ID: {gateway_config['target_id']}")
    print(f"   Region: {gateway_config['region']}")

except Exception as e:
    print(f"❌ Error registering target: {e}")
    raise

## 10. Cognito JWT 인증으로 Gateway 테스트

**목표:** Cognito JWT 토큰을 사용하여 Gateway를 테스트합니다(Lab 3, 4 및 5와 동일한 패턴).

**접근 방식:**
1. Cognito로 인증하여 JWT 토큰 가져오기
2. JWT Bearer 토큰을 사용하는 사용자 지정 HTTP MCP 클라이언트 사용
3. 엔드 투 엔드 흐름 테스트

**핵심 학습 내용:** 모든 Lab에서 일관되게 사용하는 JWT 인증 패턴

In [ ]:
# Cognito 구성 가져오기
print("🔐 Retrieving Cognito configuration...")
user_pool_id = get_parameter(PARAMETER_PATHS["cognito"]["user_pool_id"], region_name=AWS_REGION)
user_client_id = get_parameter(PARAMETER_PATHS["cognito"]["user_auth_client_id"], region_name=AWS_REGION)
test_username = get_parameter(PARAMETER_PATHS["cognito"]["test_user_email"], region_name=AWS_REGION)
test_password = get_parameter(PARAMETER_PATHS["cognito"]["test_user_password"], region_name=AWS_REGION)

print(f"  ✓ User Pool: {user_pool_id}")
print(f"  ✓ Client ID: {user_client_id}")
print(f"  ✓ Username: {test_username}")

# Cognito로 인증
print("\n🔑 Authenticating with Cognito...")
cognito = boto3.client("cognito-idp", region_name=AWS_REGION)

response = cognito.initiate_auth(
    ClientId=user_client_id,
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": test_username, "PASSWORD": test_password},
)

access_token = response["AuthenticationResult"]["AccessToken"]
id_token = response["AuthenticationResult"]["IdToken"]
expires_in = response["AuthenticationResult"]["ExpiresIn"]

print("  ✅ Authentication successful!")
print("  ✓ Token Type: Bearer")
print(f"  ✓ Expires in: {expires_in} seconds ({expires_in // 60} minutes)")
print(f"  ✓ Access Token (first 50 chars): {access_token[:50]}...")
print("\n📋 JWT tokens retrieved and ready for gateway authentication")

In [ ]:
### 10.2: JWT 토큰으로 MCP 클라이언트 초기화

from lab_helpers.lab_02.mcp_client import MCPClient

print("🔗 Initializing MCP Client for Gateway\n")

# 섹션 9에서 Gateway URL 가져오기
gateway_url = gateway_config["gateway_url"]

# JWT Bearer 토큰을 사용하는 MCP 클라이언트 생성
mcp_client = MCPClient(gateway_url, access_token)

# MCP 세션 초기화
mcp_client.initialize()

print("\n✅ MCP Client ready")
print(f"   Gateway URL: {gateway_url}")
print("   Authentication: JWT Bearer Token")
print("   Session: Initialized")
print("\n📝 Client is ready to invoke tools via MCP protocol")

In [ ]:
### 10.3: Gateway에서 사용 가능한 도구 나열

print("📋 Listing tools available on Gateway\n")

# MCP 프로토콜을 통해 도구 나열
tools = mcp_client.list_tools()

print(f"\n✅ Gateway is ready with {len(tools)} tool(s)")

### 10.3: Gateway를 통한 도구 호출 테스트

In [ ]:
### 10.4: Gateway를 통한 도구 호출 테스트

print("🧪 Testing tool invocation via Gateway MCP\n")
print("=" * 70)

# Gateway는 도구 이름 앞에 대상 접두사를 추가: "strands-diagnostics-agent___<tool_name>"
agent_tool_full = "strands-diagnostics-agent___invoke_diagnostics_agent"
test_query = "Provide a summary of critical issues with key data points and resource details."
test_args = {"query": test_query}

print(f"\n📤 Test Query: {test_query}\n")

try:
    # MCP 클라이언트를 사용하여 도구 호출
    result = mcp_client.call_tool(agent_tool_full, test_args)

    # 응답을 추출하여 표시
    if "content" in result:
        print("\n" + "=" * 70)
        print("📥 Agent Response:")
        print("=" * 70)

        for item in result["content"]:
            if item.get("type") == "text":
                text = item.get("text", "")
                # 가능한 경우 JSON 파싱
                try:
                    import json

                    parsed = json.loads(text)
                    if "response" in parsed:
                        print(f"\n{parsed['response']}")
                    else:
                        print(f"\n{json.dumps(parsed, indent=2)}")
                except json.JSONDecodeError:
                    print(f"\n{text}")

    print("\n" + "=" * 70)
    print("✅ Tool invocation test complete")

except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback

    traceback.print_exc()
    print("\n" + "=" * 70)

## 11. Lab 02 리소스 정리

**목적:** Lab 02를 새로 시작할 수 있도록 Lab 02에서 생성한 모든 리소스를 제거합니다.

**주의:** 다음 항목이 삭제됩니다.
- AgentCore Gateway
- Lambda 함수
- ECR 리포지토리
- IAM 역할
- Parameter Store 항목
- CloudWatch 로그

Lab 02를 마쳤거나 처음부터 다시 시작하려면 이 섹션을 실행합니다.

In [ ]:
### 11.1: 모든 Lab 02 리소스 정리
# 다음 단계로 Lab-03을 실행할 계획이 없는 경우에만 실행

from lab_helpers.config import AWS_REGION

# 정리 실행
# cleanup_lab_02(region_name=AWS_REGION)

## 요약: Lab 2 - 진단 에이전트 아키텍처

✅ **완료:**
1. ✓ 헬퍼 도구 - 모의 데이터 지원을 포함한 CloudWatch 로그 및 지표 검색
2. ✓ IAM 구성 - 필수 권한이 있는 Lambda 실행 역할
3. ✓ 에이전트 아키텍처 - 진단용 Strands 에이전트

**워크플로 요약:**
```
User Request
    ↓
AgentCore Gateway
    ↓
Lambda Function (ECR Container)
    ↓
Strands Agent + Tools
    ├─ fetch_ec2_logs() [mock/live]
    ├─ fetch_nginx_logs() [mock/live]
    ├─ fetch_dynamodb_logs() [mock/live]
    └─ fetch_metrics() [mock/live]
    ↓
Analysis Output
    ↓
Back to Gateway
```

**다음 단계: Lab 3 - 문제 해결 에이전트** (`Lab-03-remediation-agent.ipynb`)
- 문제 해결 작업을 위한 승인 워크플로
- 안전한 스크립트 실행을 위한 Code Interpreter
- 진단 에이전트의 분석 결과와 통합